In [1]:
import pandas as pd
from pathlib import Path

base = Path("/kaggle/input/notebooks/syedjaffarrazakazmi")
oof_paths = {
    "config1_no_augmentation": base / "03-baseline-pipeline/results/oof_predictions.csv",
    "config2_geometric_augmentation": base / "04-config2-geometric-augmentation/results/oof_predictions.csv",
    "config3_photometric_augmentation": base / "05-config3-photometric-augmentation/results/oof_predictions.csv",
    "config4_combined_augmentation": base / "06-config4-combined-augmentation/results/oof_predictions.csv",
    "config5_mixup": base / "07-config6-mixup/results/oof_predictions.csv",
    "config6_cutmix": base / "08-config6-cutmix/results/oof_predictions.csv",
    "config7_randaugment": base / "09-config7-rand-augmentation/results/oof_predictions.csv",
}

for name, p in oof_paths.items():
    print(f"{name}: {'FOUND' if p.exists() else 'MISSING'}")

first = pd.read_csv(oof_paths["config1_no_augmentation"])
print("\nColumns:", first.columns.tolist())
print(first.head(3))

config1_no_augmentation: FOUND
config2_geometric_augmentation: FOUND
config3_photometric_augmentation: FOUND
config4_combined_augmentation: FOUND
config5_mixup: FOUND
config6_cutmix: FOUND
config7_randaugment: FOUND

Columns: ['file_name', 'patient_id', 'fold', 'label', 'label_name', 'pred', 'pred_name', 'prob_Healthy', 'prob_Benign', 'prob_OPMD', 'prob_OCA']
     file_name patient_id  fold  label label_name  pred pred_name  \
0  R-01-01.jpg       R-01     3      2       OPMD     2      OPMD   
1  R-01-02.jpg       R-01     3      2       OPMD     2      OPMD   
2  R-01-03.jpg       R-01     3      1     Benign     0   Healthy   

   prob_Healthy  prob_Benign  prob_OPMD  prob_OCA  
0      0.003729     0.009032   0.986987  0.000252  
1      0.000415     0.009285   0.984925  0.005374  
2      0.999966     0.000012   0.000019  0.000002  


In [2]:
import pandas as pd

wrong_counts = {}   # file_name -> number of configs that got it wrong
patient_of = {}     # file_name -> patient_id

for cfg_name, path in oof_paths.items():
    oof = pd.read_csv(path)
    oca = oof[oof["label_name"] == "OCA"]
    for _, row in oca.iterrows():
        patient_of[row["file_name"]] = row["patient_id"]
        if row["pred_name"] != "OCA":
            wrong_counts[row["file_name"]] = wrong_counts.get(row["file_name"], 0) + 1

# Include OCA images that were never wrong, so totals add up to 129
all_oca = pd.DataFrame({"file_name": list(patient_of.keys())})
all_oca["patient_id"] = all_oca["file_name"].map(patient_of)
all_oca["times_wrong"] = all_oca["file_name"].map(wrong_counts).fillna(0).astype(int)

print(f"Total OCA images: {len(all_oca)}")
print(f"Wrong in all 7 configs: {(all_oca['times_wrong'] == 7).sum()}")
print(f"Wrong in 5+ configs:    {(all_oca['times_wrong'] >= 5).sum()}")
print(f"Never wrong:            {(all_oca['times_wrong'] == 0).sum()}")

print("\nDistribution of times_wrong (how many images were wrong in N configs):")
print(all_oca["times_wrong"].value_counts().sort_index())

# Patient-level view
by_patient = (all_oca.groupby("patient_id")
              .agg(n_images=("file_name", "count"), avg_times_wrong=("times_wrong", "mean"))
              .sort_values("avg_times_wrong", ascending=False))
print(f"\nOCA patients: {len(by_patient)}")
print("\nWorst patients (by avg times wrong):")
print(by_patient.head(10))

Total OCA images: 129
Wrong in all 7 configs: 46
Wrong in 5+ configs:    82
Never wrong:            8

Distribution of times_wrong (how many images were wrong in N configs):
times_wrong
0     8
1     7
2    17
3     6
4     9
5    12
6    24
7    46
Name: count, dtype: int64

OCA patients: 61

Worst patients (by avg times wrong):
            n_images  avg_times_wrong
patient_id                           
K-009              1              7.0
N-342              2              7.0
N-302              5              7.0
N-020              1              7.0
R-05               2              7.0
R-136              1              7.0
R-145              1              7.0
R-10               1              7.0
S-87               1              7.0
S-132              1              7.0


In [3]:
import numpy as np
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score

CLASSES = ["Healthy", "Benign", "OPMD", "OCA"]
PROB_COLS = [f"prob_{c}" for c in CLASSES]

# Reuse oof_paths from Cell 1 and add Config 8 (check this slug against your input panel)
cfg_paths = {
    1: oof_paths["config1_no_augmentation"],
    2: oof_paths["config2_geometric_augmentation"],
    3: oof_paths["config3_photometric_augmentation"],
    4: oof_paths["config4_combined_augmentation"],
    5: oof_paths["config5_mixup"],
    6: oof_paths["config6_cutmix"],
    7: oof_paths["config7_randaugment"],
    8: base / "11-focal-loss-combined/results/oof_predictions.csv",
}
names = {1: "no_aug", 2: "geometric", 3: "photometric", 4: "combined",
         5: "mixup", 6: "cutmix", 7: "randaugment", 8: "combined_focal"}

for i, p in cfg_paths.items():
    print(f"cfg{i}: {'FOUND' if p.exists() else 'MISSING'}")

dfs = {i: pd.read_csv(p).sort_values("file_name").reset_index(drop=True)
       for i, p in cfg_paths.items()}

# Sanity check: every config must cover the same images with the same labels
ref = dfs[1]
for i, d in dfs.items():
    assert (d["file_name"].values == ref["file_name"].values).all(), f"cfg{i} images differ"
    assert (d["label"].values == ref["label"].values).all(), f"cfg{i} labels differ"

y = ref["label"].values

def evaluate(probs):
    preds = probs.argmax(axis=1)
    per_class_f1 = f1_score(y, preds, average=None, labels=[0, 1, 2, 3])
    per_class_recall = recall_score(y, preds, average=None, labels=[0, 1, 2, 3])
    return {"acc": accuracy_score(y, preds),
            "macro_f1": f1_score(y, preds, average="macro"),
            "OCA_f1": per_class_f1[3],
            "OCA_recall": per_class_recall[3]}

def avg_probs(cfg_ids):
    return np.mean([dfs[i][PROB_COLS].values for i in cfg_ids], axis=0)

rows = {}
for i in names:
    rows[f"cfg{i} {names[i]}"] = evaluate(dfs[i][PROB_COLS].values)

ensembles = {
    "ENSEMBLE all 8": list(names),
    "ENSEMBLE 4,5,7,8": [4, 5, 7, 8],
    "ENSEMBLE 4,5,6,7,8": [4, 5, 6, 7, 8],
}
for label, ids in ensembles.items():
    rows[label] = evaluate(avg_probs(ids))

print(pd.DataFrame(rows).T.round(4))

# Binary view: suspicious (OPMD or OCA) vs not (Healthy or Benign)
y_bin = (y >= 2).astype(int)
def eval_binary(probs):
    p_susp = probs[:, 2] + probs[:, 3]
    pred_bin = (p_susp > 0.5).astype(int)
    return {"macro_f1": f1_score(y_bin, pred_bin, average="macro"),
            "susp_recall": recall_score(y_bin, pred_bin),
            "susp_precision": precision_score(y_bin, pred_bin)}

bin_rows = {"cfg4 combined": eval_binary(dfs[4][PROB_COLS].values),
            "ENSEMBLE 4,5,7,8": eval_binary(avg_probs([4, 5, 7, 8]))}
print("\nBinary (suspicious vs not):")
print(pd.DataFrame(bin_rows).T.round(4))

cfg1: FOUND
cfg2: FOUND
cfg3: FOUND
cfg4: FOUND
cfg5: FOUND
cfg6: FOUND
cfg7: FOUND
cfg8: FOUND
                        acc  macro_f1  OCA_f1  OCA_recall
cfg1 no_aug          0.5830    0.4893  0.2160      0.1783
cfg2 geometric       0.5880    0.5195  0.3303      0.2791
cfg3 photometric     0.5633    0.5119  0.3704      0.3488
cfg4 combined        0.5850    0.5372  0.4160      0.4031
cfg5 mixup           0.5840    0.5248  0.3422      0.3488
cfg6 cutmix          0.5947    0.5273  0.3455      0.2946
cfg7 randaugment     0.6043    0.5337  0.3468      0.3333
cfg8 combined_focal  0.5807    0.5285  0.3849      0.3566
ENSEMBLE all 8       0.6433    0.5807  0.4100      0.3178
ENSEMBLE 4,5,7,8     0.6347    0.5735  0.4144      0.3566
ENSEMBLE 4,5,6,7,8   0.6353    0.5740  0.4171      0.3411

Binary (suspicious vs not):
                  macro_f1  susp_recall  susp_precision
cfg4 combined       0.6861       0.6244          0.7215
ENSEMBLE 4,5,7,8    0.7161       0.6494          0.7584


In [4]:
cfg9_path = base / "12-convnext-tiny-combined/results/oof_predictions.csv"
print("cfg9:", "FOUND" if cfg9_path.exists() else "MISSING")

d9 = pd.read_csv(cfg9_path).sort_values("file_name").reset_index(drop=True)
assert (d9["file_name"].values == ref["file_name"].values).all()
assert (d9["label"].values == ref["label"].values).all()
dfs[9] = d9
names[9] = "convnext_tiny"

resnet_avg = avg_probs(list(range(1, 9)))          # average of the 8 ResNet50 configs
convnext = dfs[9][PROB_COLS].values

rows = {
    "cfg9 convnext_tiny": evaluate(convnext),
    "ENSEMBLE all 9 (equal weight)": evaluate(avg_probs(list(range(1, 10)))),
    "ENSEMBLE 4,9": evaluate(avg_probs([4, 9])),
    "50/50 ResNet-avg + ConvNeXt": evaluate(0.5 * resnet_avg + 0.5 * convnext),
}
print(pd.DataFrame(rows).T.round(4))

cfg9: FOUND
                                  acc  macro_f1  OCA_f1  OCA_recall
cfg9 convnext_tiny             0.6147    0.5687  0.4524      0.4419
ENSEMBLE all 9 (equal weight)  0.6483    0.5902  0.4378      0.3411
ENSEMBLE 4,9                   0.6227    0.5723  0.4416      0.3953
50/50 ResNet-avg + ConvNeXt    0.6377    0.5879  0.4622      0.4031


In [10]:
vit_path = Path("/kaggle/input/datasets/syedjaffarrazakazmi/vit-results/oof_predictions.csv")
print("vit:", "FOUND" if vit_path.exists() else "MISSING")

d10 = pd.read_csv(vit_path).sort_values("file_name").reset_index(drop=True)
assert (d10["file_name"].values == ref["file_name"].values).all()
assert (d10["label"].values == ref["label"].values).all()
dfs[10] = d10
names[10] = "vit_b16"

resnet_avg = avg_probs(list(range(1, 9)))
convnext = dfs[9][PROB_COLS].values
vit = dfs[10][PROB_COLS].values

rows = {
    "cfg10 vit_b16": evaluate(vit),
    "ConvNeXt + ViT": evaluate((convnext + vit) / 2),
    "ALL 10 equal weight": evaluate(avg_probs(list(range(1, 11)))),
    "3-way: ResNet avg + ConvNeXt + ViT": evaluate((resnet_avg + convnext + vit) / 3),
}
print(pd.DataFrame(rows).T.round(4))

vit: FOUND
                                       acc  macro_f1  OCA_f1  OCA_recall
cfg10 vit_b16                       0.6113    0.5570  0.3985      0.4109
ConvNeXt + ViT                      0.6390    0.5891  0.4545      0.4264
ALL 10 equal weight                 0.6550    0.5951  0.4356      0.3411
3-way: ResNet avg + ConvNeXt + ViT  0.6570    0.6031  0.4630      0.3876


In [15]:
nb11_log = pd.read_csv(base / "12-convnext-tiny-combined/results/results_log.csv")
vit_log = pd.read_csv("/kaggle/input/datasets/syedjaffarrazakazmi/vit-results/results_log.csv")

final_log = pd.concat([nb11_log, vit_log], ignore_index=True)
final_log.to_csv("/kaggle/working/final_results_log.csv", index=False)
print(final_log[["config_name", "backbone", "oof_accuracy", "oof_macro_f1"]])

                        config_name              backbone  oof_accuracy  \
0           config1_no_augmentation              resnet50      0.583000   
1    config2_geometric_augmentation              resnet50      0.588000   
2  config3_photometric_augmentation              resnet50      0.563333   
3     config4_combined_augmentation              resnet50      0.585000   
4                     config5_mixup              resnet50      0.584000   
5                    config6_cutmix              resnet50      0.594667   
6               config7_randaugment              resnet50      0.604333   
7            config8_combined_focal              resnet50      0.580667   
8    config9_convnext_tiny_combined         convnext_tiny      0.614667   
9         config10_vit_b16_combined  vit_base_patch16_224      0.611333   

   oof_macro_f1  
0      0.489256  
1      0.519547  
2      0.511862  
3      0.537229  
4      0.524798  
5      0.527266  
6      0.533701  
7      0.528472  
8      0.568